# Mind Quest — Group 16 (PIPS)

Mind Quest is an educational jump-and-run game about general psychology, written in Python with pygame.
A prologue introduces Open Science, three levels teach **attention**, **response inhibition** and **executive control**, and a finale against Count Hackula brings the three together.
A short questionnaire at the start and a real experiment in each level save their data as CSV files in `data/`:

| When | What | Data file |
| --- | --- | --- |
| Start of the game | questionnaire: screen time, short videos, interest in psychology, gaming, alertness | `questionnaire_*.csv` |
| Level 1: Focus Fields | child flanker task (Rueda et al., 2004), with varying spacing and number of flankers | `flanker_*.csv` |
| Level 2: Hold Your Horses | stop-signal task (Verbruggen et al., 2019) | `stopsignal_*.csv` |
| Level 3: Mission Control | 2-back task (Kirchner, 1958), Tower of London (Shallice, 1982) | `nback_*.csv`, `tower_*.csv` |
| Level 3 boss | card sorting with the Rule Keeper (Milner, 1963) | `cardsort_*.csv` |

This notebook

1. starts the game (part 1),
2. loads and summarises the questionnaire and the data of each experiment (part 2), and
3. relates the questionnaire answers to the performance in the experiments (part 3).

See `README.md` for the controls and a full description of the game.

## Setup

The game needs Python 3.12 or newer and the packages in `requirements.txt`. Uncomment the line below to install them from the notebook.
If pip cannot install pygame for your Python version (for example Python 3.14), install the compatible fork with `%pip install pygame-ce` instead.

In [ ]:
# %pip install -r requirements.txt

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

DATA_DIR = Path("data")

## Part 1: Play the game

### Before a new participant

The game remembers which levels were finished (`save/progress.json`) and the best run of each level (`ghosts/`). Both are shared by everyone who plays on the same computer, so a new participant would otherwise start with levels already unlocked and race the previous player's ghost.
Set `NEW_PARTICIPANT = True` to clear both folders before the game starts. The experiment data in `data/` are never touched.

In [ ]:
import shutil

NEW_PARTICIPANT = False

if NEW_PARTICIPANT:
    for folder in (Path("save"), Path("ghosts")):
        if folder.exists():
            shutil.rmtree(folder)
            print(f"Cleared {folder}/")

### Start the game

Running the cell below opens the game window. The game starts with a short questionnaire; after that, levels open one after the other. Play as long as you like; close the window (or quit from the menu) to return to the notebook.

* `participant_id` is an anonymous label used in the data files. Leave it as `None` to get a random label such as `p-3fa2c1`.
* `allow_admin=False` switches off the admin mode (P / F1), so challenges cannot be skipped and locked levels stay locked. Use this when you collect data.
* `ask_questionnaire=False` skips the questionnaire.
* `seed` makes the trial orders reproducible.

Set `PLAY_GAME = False` to skip the game and only analyse data that are already in `data/`.

In [ ]:
PLAY_GAME = True

if PLAY_GAME:
    from src.game import run_game

    saved_files = run_game(participant_id=None, allow_admin=False, ask_questionnaire=True)
    print(f"{len(saved_files)} data file(s) saved:")
    for path in saved_files:
        print(" ", path)

## Part 2: The experiment data

Every attempt at an experiment writes one file `data/<task>_<participant>_<timestamp>_attempt-<n>.csv` with one row per trial.
The helper below reads all files of one task into a single table. Only the **test** block is analysed; practice trials are left out.

In [ ]:
def load_task(task, data_dir=DATA_DIR):
    """Read all CSV files of one task into one DataFrame (empty if there are none)."""
    files = sorted(Path(data_dir).glob(f"{task}_*.csv"))
    if not files:
        print(f"No {task} data found in {data_dir}/")
        return pd.DataFrame()
    data = pd.concat([pd.read_csv(f).assign(file=f.name) for f in files], ignore_index=True)
    print(f"{task}: {len(files)} file(s), {len(data)} rows, participants: {', '.join(data['participant_id'].unique())}")
    return data


def test_block(data):
    """Keep only the test trials if the data have a block column."""
    if "block" in data.columns:
        return data[data["block"] == "test"].copy()
    return data

### Questionnaire

Asked once at the start of the game. Screen time is in hours per day (0–16); the other questions are rated from 1 to 5.
Every answer starts at the middle of its scale, so a participant who only presses SPACE gets the default answer on every question. Those rows are flagged as `all_defaults` and should be checked before the analysis.

In [ ]:
QUESTIONNAIRE_ITEMS = ["screen_time_hours", "short_video_frequency", "psychology_interest", "gaming_frequency", "alertness"]
DEFAULT_ANSWERS = {"screen_time_hours": 3, "short_video_frequency": 3, "psychology_interest": 3, "gaming_frequency": 3, "alertness": 3}

questionnaire = load_task("questionnaire")

if not questionnaire.empty:
    questionnaire["all_defaults"] = (questionnaire[QUESTIONNAIRE_ITEMS] == pd.Series(DEFAULT_ANSWERS)).all(axis=1)
    display(questionnaire[["participant_id", *QUESTIONNAIRE_ITEMS, "all_defaults"]])
    print(f"{questionnaire['all_defaults'].sum()} of {len(questionnaire)} participants kept every default answer.")

In [ ]:
if not questionnaire.empty:
    answered = questionnaire[~questionnaire["all_defaults"]]
    print(f"Descriptives of the {len(answered)} questionnaire(s) that were really answered:")
    display(answered[QUESTIONNAIRE_ITEMS].describe().round(2))

### Level 1: Flanker task

A row of fish is shown and the player indicates the direction of the **middle** fish. The 4, 6 or 8 flanking fish swim the same way (*congruent*), the other way (*incongruent*), or are absent (*neutral*).
The **flanker effect** is the mean correct reaction time on incongruent minus congruent trials (Eriksen & Eriksen, 1974). Trials without a response count as errors.

In [ ]:
flanker = test_block(load_task("flanker"))

if not flanker.empty:
    conditions = ["congruent", "neutral", "incongruent"]
    accuracy = flanker.pivot_table(index=["participant_id", "attempt"], columns="condition",
                                   values="correct", aggfunc="mean").reindex(columns=conditions)
    mean_rt = (flanker[flanker["correct"]]
               .pivot_table(index=["participant_id", "attempt"], columns="condition",
                            values="rt_ms", aggfunc="mean")
               .reindex(columns=conditions))
    summary = mean_rt.add_suffix("_rt_ms").round(0)
    summary["accuracy"] = flanker.groupby(["participant_id", "attempt"])["correct"].mean().round(2)
    summary["flanker_effect_ms"] = (mean_rt["incongruent"] - mean_rt["congruent"]).round(0)
    display(summary)

In [ ]:
if not flanker.empty:
    correct = flanker[flanker["correct"]]
    rt = correct.groupby("condition")["rt_ms"].agg(["mean", "sem"]).reindex(conditions)
    acc = flanker.groupby("condition")["correct"].mean().reindex(conditions)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].bar(rt.index, rt["mean"], yerr=rt["sem"].fillna(0), capsize=4, color=["#4c9f70", "#9aa5b1", "#d1603d"])
    axes[0].set(title="Mean correct RT per condition", ylabel="Reaction time (ms)")
    axes[1].bar(acc.index, acc * 100, color=["#4c9f70", "#9aa5b1", "#d1603d"])
    axes[1].set(title="Accuracy per condition", ylabel="Correct (%)", ylim=(0, 100))
    fig.suptitle("Flanker task (test trials, all attempts)")
    fig.tight_layout()
    plt.show()

#### Spacing and number of flankers

Congruent and incongruent displays vary the gap between neighbouring fish (`spacing`, 35 or 85 pixels) and the number of flanking fish (`nr_flankers`, 4, 6 or 8). Close and many flankers are expected to interfere more, so the flanker effect should be larger there.
Neutral trials have no flankers and are left out here. Files from older versions of the game do not have these columns and are skipped.

In [ ]:
has_layout = not flanker.empty and "spacing" in flanker.columns and flanker["spacing"].notna().any()

if has_layout:
    flanked = flanker[flanker["condition"] != "neutral"].dropna(subset=["spacing"]).astype({"spacing": int})
    layout_rt = (flanked[flanked["correct"]]
                 .pivot_table(index=["spacing", "nr_flankers"], columns="condition", values="rt_ms", aggfunc="mean"))
    layout = layout_rt.round(0)
    layout["accuracy"] = flanked.groupby(["spacing", "nr_flankers"])["correct"].mean().round(2)
    layout["flanker_effect_ms"] = (layout_rt.get("incongruent") - layout_rt.get("congruent")).round(0)
    display(layout)

    effect = layout["flanker_effect_ms"].unstack("spacing")
    fig, ax = plt.subplots(figsize=(7, 4))
    for spacing in effect.columns:
        ax.plot(effect.index, effect[spacing], marker="o", label=f"spacing {spacing:.0f} px")
    ax.axhline(0, color="grey", linewidth=0.8)
    ax.set(title="Flanker effect by number of flankers and spacing", xlabel="Number of flankers",
           ylabel="Incongruent - congruent RT (ms)", xticks=effect.index)
    ax.legend()
    plt.show()
else:
    print("No flanker data with spacing and nr_flankers yet.")

### Level 2: Stop-signal task

A race car points left or right (A / D). On a quarter of the trials a STOP sign follows after the stop-signal delay (SSD), and no key should be pressed. The SSD follows a staircase in steps of 50 ms.
The **stop-signal reaction time (SSRT)** is estimated with the integration method (Verbruggen et al., 2019): missed go trials get the longest go RT, the go RTs are sorted, and the RT at the rank that matches the probability of responding on stop trials minus the mean SSD is the SSRT.

In [ ]:
import math


def ssrt_integration(go, stop):
    """SSRT with the integration method, as in src/game/stop_signal_task.py."""
    if go.empty or stop.empty:
        return float("nan")
    longest = go["rt_ms"].max()
    go_rts = sorted(go["rt_ms"].fillna(longest))
    p_respond = 1 - stop["correct"].mean()
    rank = min(len(go_rts), max(1, math.ceil(p_respond * len(go_rts))))
    return go_rts[rank - 1] - stop["ssd_ms"].mean()


stop_signal = test_block(load_task("stopsignal"))

if not stop_signal.empty:
    rows = []
    for (pid, attempt), trials in stop_signal.groupby(["participant_id", "attempt"]):
        go, stop = trials[trials["trial_type"] == "go"], trials[trials["trial_type"] == "stop"]
        rows.append({
            "participant_id": pid,
            "attempt": attempt,
            "go_accuracy": go["correct"].mean(),
            "mean_go_rt_ms": go.loc[go["correct"], "rt_ms"].mean(),
            "stop_success": stop["correct"].mean(),
            "mean_ssd_ms": stop["ssd_ms"].mean(),
            "ssrt_ms": ssrt_integration(go, stop),
        })
    display(pd.DataFrame(rows).set_index(["participant_id", "attempt"]).round(2))

In [ ]:
if not stop_signal.empty:
    fig, ax = plt.subplots(figsize=(8, 4))
    for (pid, attempt), trials in stop_signal.groupby(["participant_id", "attempt"]):
        stops = trials[trials["trial_type"] == "stop"]
        ax.plot(range(1, len(stops) + 1), stops["ssd_ms"], marker="o", label=f"{pid}, attempt {attempt}")
    ax.set(title="Stop-signal delay staircase", xlabel="Stop trial", ylabel="SSD (ms)")
    ax.legend()
    plt.show()

### Level 3: 2-back task

Letters appear one by one, and the player presses SPACE when a letter equals the one two steps back. Sensitivity is summarised with **d′** using the log-linear correction (Hautus, 1995).

In [ ]:
from statistics import NormalDist


def d_prime(hits, misses, false_alarms, correct_rejections):
    """d' with the log-linear correction (Hautus, 1995), as in src/game/nback_task.py."""
    hit_rate = (hits + 0.5) / (hits + misses + 1)
    false_alarm_rate = (false_alarms + 0.5) / (false_alarms + correct_rejections + 1)
    return NormalDist().inv_cdf(hit_rate) - NormalDist().inv_cdf(false_alarm_rate)


nback = test_block(load_task("nback"))

if not nback.empty:
    rows = []
    for (pid, attempt), trials in nback.groupby(["participant_id", "attempt"]):
        target, responded = trials["is_target"], trials["responded"]
        hits, misses = (target & responded).sum(), (target & ~responded).sum()
        fas, crs = (~target & responded).sum(), (~target & ~responded).sum()
        rows.append({
            "participant_id": pid,
            "attempt": attempt,
            "accuracy": trials["correct"].mean(),
            "hit_rate": hits / (hits + misses) if hits + misses else float("nan"),
            "false_alarm_rate": fas / (fas + crs) if fas + crs else float("nan"),
            "d_prime": d_prime(hits, misses, fas, crs),
            "mean_hit_rt_ms": trials.loc[target & responded, "rt_ms"].mean(),
        })
    display(pd.DataFrame(rows).set_index(["participant_id", "attempt"]).round(2))

### Level 3: Tower of London

One row per puzzle. Planning time is the time until the first ball is lifted; *extra moves* is the number of moves above the shortest solution.

In [ ]:
tower = load_task("tower")

if not tower.empty:
    tower["extra_moves"] = tower["moves"] - tower["min_moves"]
    display(tower.groupby("min_moves")[["solved", "extra_moves", "planning_time_ms", "solution_time_ms"]].mean().round(2))

    fig, ax = plt.subplots(figsize=(6, 4))
    planning = tower.groupby("min_moves")["planning_time_ms"].mean()
    ax.bar(planning.index.astype(str), planning / 1000, color="#5b7db1")
    ax.set(title="Tower of London: planning time", xlabel="Minimum number of moves", ylabel="Planning time (s)")
    plt.show()

### Level 3 boss: Card sorting with the Rule Keeper

The hidden sorting rule (colour, shape or number) changes without warning. A **perseverative error** is a wrong choice that follows the previous rule (Milner, 1963).

In [ ]:
cardsort = load_task("cardsort")

if not cardsort.empty:
    display(cardsort.groupby(["participant_id", "attempt"]).agg(
        n_cards=("trial", "size"),
        accuracy=("correct", "mean"),
        perseverative_errors=("perseverative", "sum"),
        rule_changes=("rule", lambda rules: int((rules != rules.shift()).sum() - 1)),
    ).round(2))

## Part 3: Questionnaire and performance

The questionnaire was added so that screen habits can be related to performance in the experiments, controlling for gaming experience and fatigue.
The table below joins every participant's questionnaire with one score per experiment, from their **first** attempt (later attempts are affected by practice):

* `flanker_effect_ms`: incongruent minus congruent RT (attention)
* `ssrt_ms`: stop-signal reaction time (response inhibition; lower is better)
* `nback_d_prime`: sensitivity in the 2-back task (working memory)

Participants who kept every default answer are left out. With only a few participants, the correlations are descriptive only.

In [ ]:
def first_attempt_scores():
    """One row per participant with a score from the first attempt at each experiment."""
    scores = []
    if not flanker.empty:
        first = flanker[flanker["attempt"] == 1]
        rt = first[first["correct"]].pivot_table(index="participant_id", columns="condition", values="rt_ms", aggfunc="mean")
        if {"incongruent", "congruent"} <= set(rt.columns):
            scores.append((rt["incongruent"] - rt["congruent"]).rename("flanker_effect_ms"))
    if not stop_signal.empty:
        ssrt = {}
        for pid, trials in stop_signal[stop_signal["attempt"] == 1].groupby("participant_id"):
            ssrt[pid] = ssrt_integration(trials[trials["trial_type"] == "go"], trials[trials["trial_type"] == "stop"])
        scores.append(pd.Series(ssrt, name="ssrt_ms"))
    if not nback.empty:
        dp = {}
        for pid, trials in nback[nback["attempt"] == 1].groupby("participant_id"):
            target, responded = trials["is_target"], trials["responded"]
            dp[pid] = d_prime((target & responded).sum(), (target & ~responded).sum(),
                              (~target & responded).sum(), (~target & ~responded).sum())
        scores.append(pd.Series(dp, name="nback_d_prime"))
    return pd.concat(scores, axis=1) if scores else pd.DataFrame()


scores = first_attempt_scores()

if questionnaire.empty or scores.empty:
    print("Part 3 needs questionnaire data and at least one experiment.")
else:
    answers = (questionnaire[~questionnaire["all_defaults"]]
               .sort_values("attempt").drop_duplicates("participant_id")
               .set_index("participant_id")[QUESTIONNAIRE_ITEMS])
    combined = answers.join(scores, how="inner")
    if combined.empty:
        print("No participant has both an answered questionnaire and experiment data yet.")
    else:
        print(f"{len(combined)} participant(s) with an answered questionnaire and experiment data:")
        display(combined.round(2))
        if len(combined) >= 3:
            display(combined.corr().loc[QUESTIONNAIRE_ITEMS, scores.columns].round(2))
        else:
            print("Correlations need at least 3 participants.")

In [ ]:
if not questionnaire.empty and not scores.empty and "combined" in globals() and len(combined) >= 3:
    outcomes = [column for column in scores.columns if combined[column].notna().sum() >= 3]
    if outcomes:
        fig, axes = plt.subplots(1, len(outcomes), figsize=(5 * len(outcomes), 4), squeeze=False)
        for ax, outcome in zip(axes[0], outcomes):
            ax.scatter(combined["screen_time_hours"], combined[outcome], c=combined["gaming_frequency"], cmap="viridis")
            ax.set(xlabel="Leisure screen time (hours/day)", ylabel=outcome)
        fig.suptitle("Screen time and performance (colour: gaming frequency)")
        fig.tight_layout()
        plt.show()

## References

The full list of sources is shown in the game (home screen, "References") and is available in code:

In [ ]:
from src.game.content import REFERENCES

for reference in REFERENCES:
    print(reference, end="\n\n")